In [1]:
import os

os.listdir("/kaggle/input")


['rccgnet-paper']

In [2]:
pdf_path = "/kaggle/input/rccgnet-paper/RCCGnet.pdf"


In [5]:
!pip install -q pypdf


In [7]:
import torch
import numpy as np

from transformers import AutoTokenizer, AutoModel, pipeline
from sklearn.neighbors import NearestNeighbors
from pypdf import PdfReader


2026-01-08 13:39:19.986121: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767879560.411610      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767879560.539930      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767879561.500751      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767879561.500793      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767879561.500796      55 computation_placer.cc:177] computation placer alr

**Load embedding model**

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)


**Embedding function**

In [9]:
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.cpu().numpy()


**Load RCCGnet Paper dataset**

In [10]:
def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text() + "\n"
    return text

pdf_path = "/kaggle/input/rccgnet-paper/RCCGnet.pdf"
raw_text = load_pdf_text(pdf_path)


**text cleaning**

In [19]:
import re

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)          # remove line breaks
    text = re.sub(r'\s+', ' ', text)          # remove extra spaces
    text = re.sub(r'xa\d+', '', text)         # remove xa01, xa02 etc
    text = re.sub(r'Fig\.\s*\d+', '', text)   # remove figure refs
    text = re.sub(r'Table\s*\d+', '', text)   # remove table refs
    return text.strip()


In [20]:
raw_text = clean_text(raw_text)


**Chunk the paper**

In [21]:
def chunk_text(text, chunk_size=400):
    sentences = text.split('. ')
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


In [22]:
chunks = chunk_text(raw_text)
print("Clean chunks:", len(chunks))


Clean chunks: 180


**Generate embedding**

In [13]:
embeddings = []

for chunk in chunks:
    emb = get_embedding(chunk)
    embeddings.append(emb)

embeddings = np.vstack(embeddings)
print("Embeddings shape:", embeddings.shape)


Embeddings shape: (161, 384)


**Vector search using Scikit learn**

In [14]:
retriever = NearestNeighbors(
    n_neighbors=3,
    metric="cosine"
)

retriever.fit(embeddings)


NearestNeighbors(metric='cosine', n_neighbors=3)

**Load LLM**

In [15]:
qa_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


**Retrival function**

In [23]:
def retrieve_context(query, k=2):
    query_emb = get_embedding(query)
    distances, indices = retriever.kneighbors(query_emb, n_neighbors=k)
    return [chunks[i] for i in indices[0]]


**Answer generation**

In [24]:
def answer_question(question):
    context = retrieve_context(question)

    prompt = f"""
You are an AI research assistant.
Answer the question clearly and concisely using ONLY the information provided.
Do NOT copy text verbatim.
Summarize in simple language.

Context:
{context}

Question:
{question}

Answer (in 3–5 clear sentences):
"""

    result = qa_model(prompt)
    return result[0]["generated_text"]


**Test**

In [25]:
questions = [
    "What is RCCGNet?",
    "What dataset is introduced in the paper?",
    "What is the SCR block?",
    "Why is RCCGNet efficient?"
]

for q in questions:
    print("\nQ:", q)
    print("A:", answer_question(q))



Q: What is RCCGNet?
A: The comparison table contains a Vision Transformer (2021)31, where an image is interpreted as a sequence of patches and processed by a standard transformer encoder.

Q: What dataset is introduced in the paper?


Token indices sequence length is longer than the specified maximum sequence length for this model (865 > 512). Running this sequence through the model will result in indexing errors


A: ResNet50 (2016) IncResV2 (2016) NASNet (2018) ShuffleNet (2018) BHCNet (2019) BreastNet (2020) LiverNet (2021) ViT (2021) RCCGNet (proposed) Precision 0.7309 0.7678 0.7147 0.7897 0.8671 0.7868 0.8361 0.7415 0.9062 Recall 0.6726 0.7642 0.7362 0.7507 0.8446 0.7313 0.7979 0.7083 0.8842 F1 Score 0.6954 0.7549 0.7026 0.7579 0.8504 0.7492 0.8156 0.7170 0.8890 Accuracy 0.7384 0.7879 0.7755 0.8018 0.8823 0.7988 0.8529 0.7956 0.9009

Q: What is the SCR block?
A: Transfer learning approach End-to-end trained deep learning networks Grade ResNet50 (2016) IncResV2 (2016) NASNet (2018) ShuffleNet (2018) BHCNet (2019) BreastNet (2020) LiverNet (2021) ViT (2021) RCCGNet (proposed) Precision 0 0.8478 0.8260 0.9523 1 0.9743 0.9210 0.9523 0.9743 0.9756 1 0.8076 0.8095 0.8571 0.9523 0.8181 0.9230 0.92 0.8260 0.8846 2 0.5937 0.6 0.7666 0.6451 0.75 0.6666 0.6896 0.8421 0.9047 3 0.7272 0.75 0.4827 0.6 0.75 0.8333 0.7142 0.5666 0.7241 4 0.7037 0.6071 0.9259 0.96 0.96 1 1 0.8709 1 Overall 0.7360 0.7185 0.79